In [ ]:
from pathlib import Path
from functools import lru_cache
import glob
import os
from typing import Iterable, Optional, List, Dict
import seaborn as sns

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import mplhep
import ROOT
from IPython.display import display

# Set plot style
sns.set_style("white")
sns.set_context("notebook")
sns.set_palette("colorblind")

# -----------------------------------------------------------------------------
# Notebook configuration
# -----------------------------------------------------------------------------

# Shapes tag
SHAPES_TAG = "shapes-2026-05-07"

# Safety: limit rows when experimenting to avoid huge memory usage (None = no limit)
MAX_ROWS_PER_FILE = None

# Base path to the validation database
VALIDATION_DB_DIR = os.path.join("/work/mmolch/xyh-bbtautau-crown/bbtautau/validation_database", SHAPES_TAG)

# Target sample
TARGET_SAMPLE = "w_munu"

# List of branches to load from ROOT files (None = load all branches)
SELECTED_BRANCHES = ["bpair_pt_2", "bpair_eta_2", "bpair_btag_value_2", "id_wgt_bjet"]

# Default name of the ntuple tree in ROOT files
TREE_NAME = "ntuple"

In [ ]:
def sample_name_from_yaml(yaml_path: str) -> str:
    parts = os.path.splitext(os.path.basename(yaml_path))[0].split("_")
    if len(parts) > 2 and parts[0].isdigit():
        return "_".join(parts[2:])
    return os.path.basename(yaml_path)

def load_sample_file_map(validation_db_dir: Path) -> Dict[str, Dict[str, object]]:
    sample_file_map: Dict[str, Dict[str, object]] = {}
    for yaml_path in sorted(glob.glob(os.path.join(validation_db_dir, "*.yaml"))):
        with open(yaml_path, "r", encoding="utf-8") as handle:
            yaml_data = yaml.safe_load(handle) or {}
        root_files = list((yaml_data.get("files") or {}).keys())
        sample_name = sample_name_from_yaml(yaml_path)
        sample_file_map[sample_name] = {
            "yaml_path": yaml_path,
            "root_files": root_files,
        }
    return sample_file_map

@lru_cache(maxsize=None)
def file_has_tree(file_path: str, tree_name: str = TREE_NAME) -> bool:
    f = ROOT.TFile.Open(file_path)
    if not f or f.IsZombie():
        return False
    try:
        t = f.Get(tree_name)
        return t is not None
    finally:
        f.Close()

@lru_cache(maxsize=None)
def file_branches(file_path: str, tree_name: str = TREE_NAME) -> List[str]:
    f = ROOT.TFile.Open(file_path)
    if not f or f.IsZombie():
        return []
    try:
        t = f.Get(tree_name)
        if not t:
            return []
        return [b.GetName() for b in t.GetListOfBranches()]
    finally:
        f.Close()

def _as_pandas_from_rdf(file_path: str, branches: Optional[Iterable[str]] = None, max_rows: Optional[int] = None) -> pd.DataFrame:
    rdf = ROOT.RDataFrame(TREE_NAME, [file_path])
    if max_rows is not None:
        # RDataFrame.Range returns a new RDataFrame limited to the specified number of rows
        rdf = rdf.Range(max_rows)
    if branches is None:
        # read all branches present in this file
        branches = file_branches(file_path)
    if not branches:
        return pd.DataFrame()
    try:
        arrays = rdf.AsNumpy(list(branches))
    except Exception as exc:
        print(f"Failed to read {file_path}: {exc}")
        return pd.DataFrame()
    # Convert to pandas DataFrame; AsNumpy returns arrays (possibly 0-d for scalars)
    df = pd.DataFrame({k: np.asarray(v) for k, v in arrays.items()})
    return df

def load_sample_dataframe(file_paths: List[str], branches: Optional[Iterable[str]] = None, max_rows_per_file: Optional[int] = MAX_ROWS_PER_FILE) -> pd.DataFrame:
    # Filter files that contain the requested tree
    usable = [fp for fp in file_paths if file_has_tree(fp)]
    if not usable:
        raise RuntimeError("No usable ROOT files with the configured tree found for this sample")

    # If branches not provided, compute the union of branches across usable files
    if branches is None:
        branch_sets = [set(file_branches(fp)) for fp in usable]
        union_branches = sorted(set().union(*branch_sets)) if branch_sets else []
    else:
        union_branches = list(branches)

    dfs: List[pd.DataFrame] = []
    for fp in usable:
        file_b = set(file_branches(fp))
        use_b = [b for b in union_branches if b in file_b]
        if not use_b:
            continue
        df = _as_pandas_from_rdf(fp, use_b, max_rows=max_rows_per_file)
        if df.empty:
            continue
        dfs.append(df)
    if not dfs:
        # return empty dataframe with union_branches as columns
        return pd.DataFrame(columns=union_branches)
    out = pd.concat(dfs, ignore_index=True, sort=True)
    return out

In [ ]:
# Show available samples and let user pick one
sample_file_map = load_sample_file_map(VALIDATION_DB_DIR)
sample_names = sorted(sample_file_map)
print(f"Loaded {len(sample_names)} samples from {VALIDATION_DB_DIR}")
print("First 20 samples:")
for s in sample_names[:20]:
    info = sample_file_map[s]
    print(f"  {s}: {len(info['root_files'])} files ({os.path.basename(info['yaml_path'])})")

if TARGET_SAMPLE is None:
    raise RuntimeError("No samples available in validation DB")
print("Target sample:", TARGET_SAMPLE)

# Compute union of branches for the selected sample
root_files = sample_file_map[TARGET_SAMPLE]['root_files']
branch_sets = [set(file_branches(fp)) for fp in root_files if file_has_tree(fp)]
union_branches = sorted(set().union(*branch_sets)) if branch_sets else []
print("Available branches (first 50):", union_branches[:50])

In [ ]:
# Load the DataFrame using the selected branches setting
branches_to_use = SELECTED_BRANCHES if SELECTED_BRANCHES is not None else union_branches
print("Branches to use:", branches_to_use)
df = load_sample_dataframe(root_files, branches=branches_to_use, max_rows_per_file=MAX_ROWS_PER_FILE)
print("Loaded DataFrame shape:", df.shape)
display(df.head())
display(df.info())

In [ ]:
h = np.histogram(df[~np.isfinite(df['id_wgt_bjet'])]["bpair_pt_2"], bins=100, range=(0, 400))
mplhep.histplot(h)
plt.xlabel("bpair_pt_2")
plt.ylabel("Events")

In [ ]:
h = np.histogram(df[~np.isfinite(df['id_wgt_bjet'])]["bpair_eta_2"], bins=100, range=(-2.5, 2.5))
mplhep.histplot(h)
plt.xlabel("bpair_eta_2")
plt.ylabel("Events")

In [ ]:
h = np.histogram(df[~np.isfinite(df['id_wgt_bjet'])]["bpair_btag_value_2"], bins=100, range=(-1, 1))
mplhep.histplot(h)
plt.xlabel("bpair_btag_value_2")
plt.ylabel("Events")